In [ ]:
from datasets import load_dataset, concatenate_datasets
from trl import SFTConfig, SFTTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import math

model_id = 'model'
assistant_model_id = 'assistant'


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(assistant_model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

model_size = sum(t.numel() for t in model.parameters())
print(f"Transformer size: {model_size/1000**2:.1f}M parameters")

In [ ]:
from datasets import load_dataset, concatenate_datasets

alpaca = load_dataset("yahma/alpaca-cleaned", split="train")
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")

def format_alpaca(examples):
    messages = []
    for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        user_msg = instr.strip()
        if inp and inp.strip():
            user_msg += "\n\n" + inp.strip()

        messages.append([
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": out.strip()},
        ])
    return {"messages": messages}

def format_dolly(examples):
    messages = []
    for instr, ctx, out in zip(examples["instruction"], examples["context"], examples["response"]):
        user_msg = instr.strip()
        if ctx and ctx.strip():
            user_msg += "\n\n" + ctx.strip()

        messages.append([
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": out.strip()},
        ])
    return {"messages": messages}

alpaca_fmt = alpaca.map(
    format_alpaca,
    batched=True,
    remove_columns=alpaca.column_names,
    num_proc=4,
)

dolly_fmt = dolly.map(
    format_dolly,
    batched=True,
    remove_columns=dolly.column_names,
    num_proc=4,
)

raw_datasets = concatenate_datasets([
    alpaca_fmt,
    dolly_fmt,
]).shuffle(seed=42).train_test_split(test_size=0.1)

print(raw_datasets)

In [ ]:
print(tokenizer.apply_chat_template(raw_datasets['test'][0]['messages'], tokenize=False))

In [ ]:

training_args = SFTConfig(
    output_dir=assistant_model_id,
    overwrite_output_dir=True,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=500,
    save_steps=500,
    save_total_limit=2,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    weight_decay=0.0,
    fp16=True,
    bf16=False,
    report_to=[],
    max_length=512,
    packing=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    assistant_only_loss=True,
)

class PerplexitySFTTrainer(SFTTrainer):
    def log(self, logs, *args, **kwargs):
        if "eval_loss" in logs:
            logs["eval_perplexity"] = round(math.exp(logs["eval_loss"]), 2)
        super().log(logs, *args, **kwargs)

trainer = PerplexitySFTTrainer(
    processing_class=tokenizer,
    model=model,
    args=training_args,
    train_dataset=raw_datasets['train'],
    eval_dataset=raw_datasets['test'],
)


In [ ]:
trainer.train()

In [ ]:
trainer.save_model(assistant_model_id)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
logs = trainer.state.log_history

train_steps = [log["step"] for log in logs if "loss" in log and "eval_loss" not in log]
train_losses = [log["loss"] for log in logs if "loss" in log and "eval_loss" not in log]

eval_steps = [log["step"] for log in logs if "eval_loss" in log]
eval_losses = [log["eval_loss"] for log in logs if "eval_loss" in log]

plt.figure(figsize=(12, 6))
plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Validation Loss", linewidth=2)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()